## 1. Kurulum ve Bağımlılıklar

## ⚙️ Konfigürasyon — Sadece bu hücreyi değiştir!

Farklı bir branch veya dataset için **yalnızca aşağıdaki hücreyi** güncelle. Geri kalan tüm hücreler buradan okur.

In [ ]:
REPO_URL        = "https://github.com/Aliekinozcetin/DiffuVQA.git"
BRANCH          = "Bert"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DiffuVQA"
LOCAL_CLONE_PATH   = "/content/DiffuVQA"

MODEL_NAME      = "bert"
MODEL_LABEL     = "DiffuVQA-Bert"

DATASET         = "SLAKE"
DATA_DIR        = "datasets"
IMAGEFOLDER_NAME = "SLAKE/imgs"
DRIVE_IMAGE_DIR = f"{DRIVE_PROJECT_PATH}/datasets/SLAKE/imgs"
LOCAL_IMAGE_DIR = f"{LOCAL_CLONE_PATH}/datasets/SLAKE/imgs"

DATASET_IMG_PATHS = [
    f"/content/drive/MyDrive/datasets/SLAKE/imgs",
    f"/content/drive/MyDrive/DiffuVQA/datasets/SLAKE/imgs",
    f"/content/drive/Drive'ım/DiffuVQA/datasets/SLAKE/imgs",
]

BATCH_SIZE      = 64
MICROBATCH      = 16
LR              = 0.00005

LEARNING_STEPS  = 200000
DIFFUSION_STEPS = 2000
SEQ_LEN         = 32
HIDDEN_T_DIM    = 768
SEED            = 42
SAVE_INTERVAL   = 2500
LOG_INTERVAL    = 500

USE_NOISING_F          = False
PRE_ANSWER_LOSS_WEIGHT = 0.0

SAMPLE_STEP     = 200

CHECKPOINT_PATH = "/content/drive/MyDrive/DiffuVQA/checkpoints/slake/bert/btch64/lr5e-05"
SAMPLE_FOLDER   = f"/content/drive/MyDrive/DiffuVQA/samples/{DATASET.lower()}/{MODEL_NAME.lower()}"
OUTPUT_CSV      = f"./reports/{MODEL_NAME}_{DATASET.lower()}_evaluation_results.csv"
DRIVE_RESULTS_PATH = "/content/drive/MyDrive/DiffuVQA_Results/"

# Sıfırdan eğitim: None bırak. Devam: tam yol gir.
RESUME_CHECKPOINT = f"{CHECKPOINT_PATH}/ema_0.9999_025000.pt"

print(f"Branch    : {BRANCH}  →  {REPO_URL}")
print(f"Model     : {MODEL_NAME} | Dataset: {DATASET}")
print(f"Batch: {BATCH_SIZE} | Microbatch: {MICROBATCH} | LR: {LR}")
print(f"Steps: {LEARNING_STEPS} | DiffSteps: {DIFFUSION_STEPS} | SeqLen: {SEQ_LEN}")
print(f"Save: {SAVE_INTERVAL} | Log: {LOG_INTERVAL} | Sample: {SAMPLE_STEP} (DDIM)")
print(f"use_noising_f: {USE_NOISING_F} | pre_answer_loss_weight: {PRE_ANSWER_LOSS_WEIGHT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Resume    : {RESUME_CHECKPOINT or 'None (sıfırdan eğitim)'}")

### GPU Kontrolü

In [20]:
!nvidia-smi

Tue May 19 19:50:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             46W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Google Drive Bağlantısı

Checkpoint'ler ve dataset Drive'da kalıcı olarak saklanır.

In [21]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Repo Clone + Drive Sync

GitHub'dan taze klon çeker, kod dosyalarını Drive'a kopyalar (`datasets/`, `checkpoints/` atlanır).

In [22]:
import os
import shutil

os.chdir("/content")

if os.path.exists(LOCAL_CLONE_PATH):
    print(f"Eski geçici klasör siliniyor: {LOCAL_CLONE_PATH}")
    shutil.rmtree(LOCAL_CLONE_PATH)

print("GitHub'dan taze kopya çekiliyor...")
!git clone -b {BRANCH} --single-branch {REPO_URL} {LOCAL_CLONE_PATH}

print(f"Drive'daki kod dosyaları güncelleniyor: {DRIVE_PROJECT_PATH}")
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH)

SKIP_DIRS = {"datasets", "checkpoints", "samples", "outputs", "reports", ".git"}
for item in os.listdir(LOCAL_CLONE_PATH):
    if item in SKIP_DIRS:
        continue
    src_path = os.path.join(LOCAL_CLONE_PATH, item)
    dst_path = os.path.join(DRIVE_PROJECT_PATH, item)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.chdir(DRIVE_PROJECT_PATH)
print(f"Çalışma dizini: {os.getcwd()}")

Eski geçici klasör siliniyor: /content/DiffuVQA
GitHub'dan taze kopya çekiliyor...
Cloning into '/content/DiffuVQA'...
remote: Enumerating objects: 1168, done.
remote: Counting objects: 100% (553/553), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 1168 (delta 444), reused 417 (delta 378), pack-reused 615 (from 2)
Receiving objects: 100% (1168/1168), 109.86 MiB | 19.82 MiB/s, done.
Resolving deltas: 100% (577/577), done.
Drive'daki kod dosyaları güncelleniyor: /content/drive/MyDrive/DiffuVQA
Çalışma dizini: /content/drive/MyDrive/DiffuVQA


### Mevcut Drive Reposunu Güncelle

Clone yerine Drive'daki mevcut repoyu `git reset --hard` ile güncel branch'e çeker. Clone hücresine alternatif.

In [ ]:
if os.path.exists(DRIVE_PROJECT_PATH):
    os.chdir(DRIVE_PROJECT_PATH)
    print(f"Çalışma dizini: {os.getcwd()}")
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}
    !git log -3 --oneline
else:
    print("Drive'da proje klasörü bulunamadı. Önce clone hücresini çalıştır.")

Çalışma dizini: /content/drive/MyDrive/DiffuVQA


### Dataset Görüntülerini Kopyala

Drive'daki veya yerel addres listesindeki SLAKE görüntülerini Colab runtime'ına kopyalar. `ACTIVE_IMAGE_DIR` bu hücrede belirlenir.

In [ ]:
import os
import shutil
from tqdm.auto import tqdm

def resolve_dataset_image_source(candidate_paths):
    seen = set()
    for path in candidate_paths:
        if not path or not isinstance(path, str):
            continue
        norm = os.path.normpath(path)
        if norm in seen:
            continue
        seen.add(norm)
        if os.path.exists(path):
            return path
    return None

# IMAGEFOLDER_NAME zaten tam yolu içeriyor (örn. "SLAKE/imgs")
dataset_local_imgs = os.path.join(LOCAL_CLONE_PATH, DATA_DIR, IMAGEFOLDER_NAME)

source_candidates = [
    LOCAL_IMAGE_DIR,
    DRIVE_IMAGE_DIR,
    *DATASET_IMG_PATHS,
]
source_image_path = resolve_dataset_image_source(source_candidates)

ACTIVE_IMAGE_DIR = dataset_local_imgs

if source_image_path:
    src_abs = os.path.abspath(source_image_path)
    dst_abs = os.path.abspath(dataset_local_imgs)

    if src_abs != dst_abs:
        os.makedirs(dataset_local_imgs, exist_ok=True)
        files_to_copy = []
        for root, _, files in os.walk(source_image_path):
            for file_name in files:
                src_file = os.path.join(root, file_name)
                rel_path = os.path.relpath(src_file, source_image_path)
                dst_file = os.path.join(dataset_local_imgs, rel_path)
                files_to_copy.append((src_file, dst_file))

        for src_file, dst_file in tqdm(files_to_copy, desc="Copying dataset images", unit="file"):
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            shutil.copy2(src_file, dst_file)

        print(f"Copied {len(files_to_copy)} files → {dataset_local_imgs}")
    else:
        print("Source already at local runtime path; skipping copy.")
else:
    print("No image source found; using existing local directory if present.")

print(f"ACTIVE_IMAGE_DIR: {ACTIVE_IMAGE_DIR}")

### Dataset Doğrulama

JSONL dosyalarının ve görüntü klasörünün doğru yerde olduğunu kontrol eder.

In [ ]:
import os

!echo "Dataset klasörü:"
!ls -lh datasets/
!echo "JSONL dosyaları:"
!ls -lh datasets/*.jsonl
print(f"Active image dir: {ACTIVE_IMAGE_DIR}")
if os.path.exists(ACTIVE_IMAGE_DIR):
    print("İlk 5 dosya:")
    for i, name in enumerate(sorted(os.listdir(ACTIVE_IMAGE_DIR))[:5], 1):
        print(f"  {i}. {name}")
else:
    print("Active image dir bulunamadı.")
!echo "Train set örneği:"
!head -n 1 datasets/train.jsonl

### Görüntü Önizleme

Rastgele 5 dataset görüntüsünü gösterir. Kopyalama başarılıysa burada görüntüler çıkmalı.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import random

image_extensions = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".tiff")
all_image_files = []
for root, _, files in os.walk(ACTIVE_IMAGE_DIR):
    for file in files:
        if file.lower().endswith(image_extensions):
            all_image_files.append(os.path.join(root, file))

if not all_image_files:
    print(f"No images found in {ACTIVE_IMAGE_DIR}.")
else:
    print(f"Found {len(all_image_files)} images.")
    num_examples = min(5, len(all_image_files))
    selected_images = random.sample(all_image_files, num_examples)

    plt.figure(figsize=(15, 5))
    for i, img_path in enumerate(selected_images):
        try:
            img = Image.open(img_path)
            plt.subplot(1, num_examples, i + 1)
            plt.imshow(img)
            plt.title(os.path.basename(img_path), fontsize=8)
            plt.axis("off")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
    plt.tight_layout()
    plt.show()

### Bağımlılıkları Kur

In [ ]:
!pip install --upgrade pip -q
!pip install -r requirements_colab.txt -q
!pip install pandas openpyxl bert-score -q
!python -m spacy download en_core_web_sm -q
print("Kurulum tamamlandı.")

### Python Import'ları

In [ ]:
import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict
import glob

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Model Eğitimi (BERT)

### BERT Ağırlıklarını Cache'e Al

HF mirror BERT'i servis etmediğinden `HF_ENDPOINT` geçici kaldırılır, ağırlıklar indirilir, sonra geri eklenir.

In [ ]:
import os
from transformers import BertModel, BertTokenizer

# HF_ENDPOINT hf-mirror BERT ağırlıklarını servis etmiyor; geçici olarak kaldırılıyor
_endpoint = os.environ.pop("HF_ENDPOINT", None)
BertTokenizer.from_pretrained("bert-base-uncased")
BertModel.from_pretrained("bert-base-uncased")
if _endpoint:
    os.environ["HF_ENDPOINT"] = _endpoint
print("BERT cache'e alındı.")

### Eğitimi Başlat

`RESUME_CHECKPOINT` set ise kaldığı yerden devam eder. Checkpoint'ler `CHECKPOINT_PATH`'e her `SAVE_INTERVAL` adımda kaydedilir.

In [ ]:
import os
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

resume_flag = f"--resume_checkpoint {RESUME_CHECKPOINT}" if RESUME_CHECKPOINT else ""

!python train.py \
    --vocab {MODEL_NAME.lower()} \
    --use_plm_init {MODEL_NAME} \
    --batch_size {BATCH_SIZE} \
    --microbatch {MICROBATCH} \
    --lr {LR} \
    --diffusion_steps {DIFFUSION_STEPS} \
    --seq_len {SEQ_LEN} \
    --hidden_t_dim {HIDDEN_T_DIM} \
    --checkpoint_path {CHECKPOINT_PATH} \
    --seed {SEED} \
    --learning_steps {LEARNING_STEPS} \
    --save_interval {SAVE_INTERVAL} \
    --log_interval {LOG_INTERVAL} \
    --eval_interval 500 \
    --data_dir {DATA_DIR} \
    --dataset {DATASET} \
    --image_dir {ACTIVE_IMAGE_DIR} \
    --use_noising_f {USE_NOISING_F} \
    --pre_answer_loss_weight {PRE_ANSWER_LOSS_WEIGHT} \
    {resume_flag}

## 3. Model Örnekleme (Inference)

### Sampling — Test Seti Üzerinde Cevap Üret

En son checkpoint ile `sample_vqa_GPU.py` çalıştırır. `SAMPLE_STEP=200` DDIM ile 2000 adım 10x hızlanır. Çıktı `SAMPLE_FOLDER`'a `.jsonl` olarak kaydedilir.

In [ ]:
import glob

SAMPLE_SEED2 = 102

checkpoint_files = sorted(glob.glob(f"{CHECKPOINT_PATH}/ema_*.pt"))
if checkpoint_files:
    checkpoint_file = checkpoint_files[-1]
    print(f"Checkpoint: {checkpoint_file}")
else:
    print("Checkpoint bulunamadı. Önce eğitimi tamamlayın.")
    checkpoint_file = None

if checkpoint_file:
    !python sample_vqa_GPU.py \
        --model_path {checkpoint_file} \
        --step {SAMPLE_STEP} \
        --batch_size {BATCH_SIZE} \
        --seed2 {SAMPLE_SEED2} \
        --split test \
        --out_dir {SAMPLE_FOLDER} \
        --clamp_step 0

    !ls -lh {SAMPLE_FOLDER}/*.jsonl | tail -5

## 4. Model Değerlendirme ve CSV Export

### Hızlı Eval — eval_DiffuVQA.py

Son checkpoint'in sample dosyasını bulur, ilk 5 örneği gösterir ve `eval_DiffuVQA.py` ile BLEU/ROUGE/METEOR/BERTScore/CIDEr hesaplar.

In [ ]:
import glob, json, os

checkpoint_files = sorted(glob.glob(f"{CHECKPOINT_PATH}/ema_*.pt"))
if checkpoint_files:
    _ckpt_name = os.path.basename(checkpoint_files[-1])
    CURRENT_SAMPLE_FILENAME = f"{_ckpt_name}.seed{SAMPLE_SEED2}_step0_samplestep{SAMPLE_STEP}_bsize{BATCH_SIZE}.jsonl"
else:
    CURRENT_SAMPLE_FILENAME = f"ema_0.9999_{LEARNING_STEPS:06d}.pt.seed{SAMPLE_SEED2}_step0_samplestep{SAMPLE_STEP}_bsize{BATCH_SIZE}.jsonl"

print(f"Değerlendirilecek dosya: {CURRENT_SAMPLE_FILENAME}")

sample_path = os.path.join(SAMPLE_FOLDER, CURRENT_SAMPLE_FILENAME)
if os.path.exists(sample_path):
    with open(sample_path) as f:
        rows = [json.loads(l) for l in f][:5]
    print("\nİlk 5 örnek:")
    for i, r in enumerate(rows):
        gen = r.get('generate_answer', '')
        ref = r.get('reference_answer', '')
        match = "AYNI (leakage?)" if gen.strip() == ref.strip() else "farklı"
        print(f"  [{i+1}] {match} | ref: '{ref}' | gen: '{gen}'")

!python eval/eval_DiffuVQA.py --folder {SAMPLE_FOLDER} --filename {CURRENT_SAMPLE_FILENAME}

### Metrik Fonksiyonları

BLEU, ROUGE-L, METEOR, CIDEr hesaplama fonksiyonları. `evaluate_and_export_csv` tarafından kullanılır.

In [ ]:
from torchmetrics.text.rouge import ROUGEScore
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

rougeScore = ROUGEScore()

def get_bleu(recover, reference, n=1):
    weights = tuple((1.0 / n for _ in range(n)))
    return sentence_bleu([reference.split()], recover.split(),
                         weights=weights,
                         smoothing_function=SmoothingFunction().method4)

def rouge_l_score(prediction, reference):
    scores = rougeScore(prediction, reference)
    return scores['rougeL_fmeasure'].item()

def calculate_meteor(prediction, reference):
    return meteor_score([word_tokenize(reference)], word_tokenize(prediction))

def cider_score(predictions, references):
    if not predictions or not references:
        return 0.0
    try:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(predictions + references)
        n_pred = len(predictions)
        similarities = cosine_similarity(tfidf_matrix[:n_pred], tfidf_matrix[n_pred:])
        return similarities.diagonal().mean()
    except Exception:
        return 0.0

print("Değerlendirme fonksiyonları hazır.")

### CSV Export Fonksiyonu

Tüm sample dosyaları için metrik hesaplar, sonuçları CSV'ye yazar. BERTScore için DeBERTa modeli yüklenir.

In [ ]:
def evaluate_and_export_csv(sample_files, output_csv="evaluation_results.csv",
                            dataset_file="datasets/test.jsonl",
                            model_label="DiffuVQA", dataset_label="SLAKE"):
    if isinstance(sample_files, str):
        sample_files = [sample_files]

    answer_type_map = {}
    try:
        with open(dataset_file, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                qid = data.get('qid')
                question = data.get('question', '').strip().lower()
                answer_type = data.get('answer_type', 'UNKNOWN')
                if qid:
                    answer_type_map[qid] = answer_type
                if question:
                    answer_type_map[question] = answer_type
    except Exception as e:
        print(f"Dataset yüklenemedi: {e}")

    all_results = []

    for sample_file in sample_files:
        print(f"Değerlendiriliyor: {sample_file}")
        samples = []
        with open(sample_file, 'r', encoding='utf-8') as f:
            for line in f:
                samples.append(json.loads(line))

        bleu1_scores, rougeL_scores, meteor_scores, f1_scores = [], [], [], []
        predictions, references = [], []
        total_samples = correct_all = correct_yn = correct_oe = 0
        count_yn = count_oe = empty_count = 0

        for sample in samples:
            pred = (sample.get('generate_answer') or sample.get('recover') or
                    sample.get('generated_answer') or sample.get('prediction') or '').strip()
            ref  = (sample.get('reference_answer') or sample.get('reference') or
                    sample.get('answer') or '').strip()
            if not ref:
                continue

            q_type = sample.get('answer_type', 'UNKNOWN')
            if q_type == 'UNKNOWN':
                qid = sample.get('qid')
                question = sample.get('question', '').strip().lower()
                q_type = answer_type_map.get(qid) or answer_type_map.get(question, 'UNKNOWN')

            if not pred:
                empty_count += 1
                pred = "[EMPTY]"

            predictions.append(pred)
            references.append(ref)
            total_samples += 1

            em = pred.lower().strip() == ref.lower().strip()
            if em:
                correct_all += 1
                if q_type == 'CLOSED': correct_yn += 1
                elif q_type == 'OPEN':  correct_oe += 1
            if q_type == 'CLOSED': count_yn += 1
            elif q_type == 'OPEN':  count_oe += 1

            if pred != "[EMPTY]":
                pred_tokens = set(pred.lower().split())
                ref_tokens  = set(ref.lower().split())
                if pred_tokens and ref_tokens:
                    precision = len(pred_tokens & ref_tokens) / len(pred_tokens)
                    recall    = len(pred_tokens & ref_tokens) / len(ref_tokens)
                    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                else:
                    f1 = 0.0
                f1_scores.append(f1)
                bleu1_scores.append(get_bleu(pred, ref, n=1))
                rougeL_scores.append(rouge_l_score(pred, ref))
                meteor_scores.append(calculate_meteor(pred, ref))
            else:
                f1_scores.append(0.0)
                bleu1_scores.append(0.0)
                rougeL_scores.append(0.0)
                meteor_scores.append(0.0)

        cider = cider_score(predictions, references)

        bert_f1_score = 0.0
        try:
            import warnings, logging
            from bert_score import score as bert_score_fn
            valid_preds = [p for p in predictions if p != "[EMPTY]"]
            valid_refs  = [r for i, r in enumerate(references) if predictions[i] != "[EMPTY]"]
            if valid_preds and valid_refs:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    logging.disable(logging.WARNING)
                    _, _, bert_f1 = bert_score_fn(
                        valid_preds, valid_refs,
                        lang='en', verbose=False,
                        device='cuda' if torch.cuda.is_available() else 'cpu'
                    )
                    logging.disable(logging.NOTSET)
                bert_f1_score = bert_f1.mean().item()
        except Exception as e:
            print(f"BERTScore hesaplanamadı: {e}")

        results = {
            'model_name': model_label,
            'dataset_name': dataset_label,
            'export_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'overall_accuracy': correct_all / total_samples if total_samples > 0 else 0.0,
            'yes_no_accuracy': correct_yn / count_yn if count_yn > 0 else 0.0,
            'open_ended_accuracy': correct_oe / count_oe if count_oe > 0 else 0.0,
            'bleu_1_score': np.mean(bleu1_scores) if bleu1_scores else 0.0,
            'rouge_l_score': np.mean(rougeL_scores) if rougeL_scores else 0.0,
            'meteor_score': np.mean(meteor_scores) if meteor_scores else 0.0,
            'cider_score': cider,
            'bert_score': bert_f1_score,
            'f1_score': np.mean(f1_scores) if f1_scores else 0.0,
            'additional_info': f"Empty: {empty_count} ({100*empty_count/total_samples:.1f}%)" if total_samples > 0 else "",
            'Sample Folder': os.path.dirname(sample_file) or "./samples",
            'Total Samples': total_samples,
        }
        all_results.append(results)

        print(f"  Total: {total_samples} | CLOSED: {count_yn} | OPEN: {count_oe}")
        print(f"  overall_acc: {results['overall_accuracy']:.4f} | f1: {results['f1_score']:.4f}")
        print(f"  BLEU-1: {results['bleu_1_score']:.4f} | ROUGE-L: {results['rouge_l_score']:.4f}")
        print(f"  METEOR: {results['meteor_score']:.4f} | CIDEr: {results['cider_score']:.4f}")
        print(f"  BERTScore: {results['bert_score']:.4f}")

    df = pd.DataFrame(all_results)
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nSonuçlar kaydedildi: {output_csv}")
    return df

print("CSV export fonksiyonu hazır.")

### Tüm Sample Dosyalarını Değerlendir ve CSV'ye Kaydet

In [ ]:
sample_files = glob.glob(f"{SAMPLE_FOLDER}/*.jsonl")
os.makedirs("./reports", exist_ok=True)

if not sample_files:
    print(f"Örnek dosyası bulunamadı: {SAMPLE_FOLDER}")
else:
    print(f"{len(sample_files)} örnek dosyası bulundu")
    results_df = evaluate_and_export_csv(
        sample_files,
        output_csv=OUTPUT_CSV,
        model_label=MODEL_LABEL,
        dataset_label=DATASET,
    )
    display(results_df)


## 5. Sonuçları İndir

### CSV'yi Drive'a Kaydet ve İndir

In [ ]:
from google.colab import files

os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)

if os.path.exists(OUTPUT_CSV):
    shutil.copy(OUTPUT_CSV, DRIVE_RESULTS_PATH)
    print(f"CSV kaydedildi: {DRIVE_RESULTS_PATH}")
    files.download(OUTPUT_CSV)
else:
    print(f"CSV bulunamadı: {OUTPUT_CSV}")


### Sonuçları Görselleştir

Accuracy ve NLG metriklerini bar chart olarak çizer, `./reports/metrics_visualization.png` olarak kaydeder.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'results_df' in locals() and len(results_df) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    accuracy_cols = ['overall_accuracy', 'yes_no_accuracy', 'open_ended_accuracy']
    results_df[accuracy_cols].iloc[0].plot(kind='bar', ax=axes[0], color='skyblue')
    axes[0].set_title('Accuracy Metrics (CLOSED=Yes/No, OPEN=Open-Ended)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_ylim([0, 1])
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

    nlg_cols = ['f1_score', 'bleu_1_score', 'rouge_l_score', 'meteor_score', 'cider_score', 'bert_score']
    results_df[nlg_cols].iloc[0].plot(kind='bar', ax=axes[1], color='lightcoral')
    axes[1].set_title('NLG & Semantic Similarity Metrics', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Score', fontsize=12)
    axes[1].set_ylim([0, 1])
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig('./reports/metrics_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Görselleştirme kaydedildi: ./reports/metrics_visualization.png")
else:
    print("Görselleştirme için sonuç bulunamadı.")
